In [1]:
"""


USAGE:
  python dataset_comparison.py

"""
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from scipy import stats
from scipy.stats import mannwhitneyu, ks_2samp
import shap

# ── Try UMAP (optional) ───────────────────────────────────────
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("  [INFO] umap-learn not installed — skipping UMAP plot")
    print("         Install with: pip install umap-learn")

# ─────────────────────────────────────────────────────────────
# PATHS  ← edit these
# ─────────────────────────────────────────────────────────────
WESTERN_PATH = "C:/Users/tanvi/Downloads/shapfeatures.csv"
INDIAN_PATH  = "C:/Users/tanvi/Downloads/indian_AG.csv"
OUT_DIR      = "C:/Users/tanvi/Downloads/dataset_comparison"
# ── Indian dataset column names — adjust if yours differ ─────
INDIAN_LABEL_COL   = 'label'       # column name for labels in Indian CSV
INDIAN_DATASET_COL = 'dataset_id'  # if present, will be dropped

os.makedirs(OUT_DIR, exist_ok=True)

BLUE   = '#4f8ef7'
RED    = '#ef4444'
GREEN  = '#10b981'
PURPLE = '#7c3aed'
AMBER  = '#f59e0b'
GRAY   = '#6b7a99'

def save(name):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close('all')
    print(f"  → saved: {name}")
    return path

# ═══════════════════════════════════════════════════════════════
# 1. LOAD DATASETS
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1 — LOADING DATASETS")
print("=" * 60)

# ── Western ───────────────────────────────────────────────────
df_w = pd.read_csv(WESTERN_PATH)
df_w = df_w.dropna().loc[:, ~df_w.columns.duplicated()]

DROP_W   = [c for c in ['Unnamed: 0', 'label', 'dataset_id'] if c in df_w.columns]
feat_w   = [c for c in df_w.columns if c not in DROP_W]
X_w      = df_w[feat_w].values.astype(np.float32)
y_w      = LabelEncoder().fit_transform(df_w['label'].values)

print(f"Western  — samples: {X_w.shape[0]}  genes: {X_w.shape[1]}")
print(f"  Tumor(1): {(y_w==1).sum()}  Normal(0): {(y_w==0).sum()}")

# ── Indian ────────────────────────────────────────────────────
df_i = pd.read_csv(INDIAN_PATH)
df_i = df_i.dropna().loc[:, ~df_i.columns.duplicated()]

DROP_I   = [c for c in ['Unnamed: 0', INDIAN_LABEL_COL, INDIAN_DATASET_COL] if c in df_i.columns]
feat_i   = [c for c in df_i.columns if c not in DROP_I]

# ── Find common genes ─────────────────────────────────────────
common_genes = sorted(set(feat_w) & set(feat_i))
print(f"\nIndian   — samples: {df_i.shape[0]}  genes in file: {len(feat_i)}")
print(f"Common genes (intersection): {len(common_genes)}")

if len(common_genes) == 0:
    print("\n[ERROR] No common genes found. Check that column names match between datasets.")
    print(f"  Western sample cols: {feat_w[:5]}")
    print(f"  Indian  sample cols: {feat_i[:5]}")
    sys.exit(1)

X_w_c  = df_w[common_genes].values.astype(np.float32)
X_i_c  = df_i[common_genes].values.astype(np.float32)

if INDIAN_LABEL_COL in df_i.columns:
    y_i = LabelEncoder().fit_transform(df_i[INDIAN_LABEL_COL].values)
else:
    y_i = np.ones(len(df_i), dtype=int)   # assume all tumor if no label
    print("  [WARN] No label column found in Indian dataset — treating all as tumor")

print(f"Indian   — Tumor(1): {(y_i==1).sum()}  Normal(0): {(y_i==0).sum()}")

N_GENES = len(common_genes)

# ── Summary dict (will be written to JSON for the Word doc) ──
summary = {
    'western_samples'  : int(X_w_c.shape[0]),
    'western_tumor'    : int((y_w==1).sum()),
    'western_normal'   : int((y_w==0).sum()),
    'indian_samples'   : int(X_i_c.shape[0]),
    'indian_tumor'     : int((y_i==1).sum()),
    'indian_normal'    : int((y_i==0).sum()),
    'common_genes'     : N_GENES,
    'western_only_genes': int(len(set(feat_w) - set(feat_i))),
    'indian_only_genes' : int(len(set(feat_i) - set(feat_w))),
}

# ═══════════════════════════════════════════════════════════════
# 2. BASIC STATISTICS
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 2 — BASIC EXPRESSION STATISTICS")
print("=" * 60)

mean_w = X_w_c.mean(axis=0)
mean_i = X_i_c.mean(axis=0)
std_w  = X_w_c.std(axis=0)
std_i  = X_i_c.std(axis=0)

summary['western_global_mean'] = round(float(mean_w.mean()), 4)
summary['indian_global_mean']  = round(float(mean_i.mean()), 4)
summary['western_global_std']  = round(float(std_w.mean()),  4)
summary['indian_global_std']   = round(float(std_i.mean()),  4)

# ── Plot 2a: Mean expression scatter ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Scatter: western mean vs indian mean per gene
axes[0].scatter(mean_w, mean_i, alpha=0.3, s=8, color=PURPLE)
lim = [min(mean_w.min(), mean_i.min()), max(mean_w.max(), mean_i.max())]
axes[0].plot(lim, lim, 'r--', lw=1, label='y=x (identical)')
axes[0].set_xlabel('Western mean expression', fontsize=11)
axes[0].set_ylabel('Indian mean expression', fontsize=11)
axes[0].set_title('Mean Gene Expression\nWestern vs Indian (per gene)', fontweight='bold')
axes[0].legend(fontsize=9)

corr_means = float(np.corrcoef(mean_w, mean_i)[0, 1])
axes[0].text(0.05, 0.92, f'r = {corr_means:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             color=PURPLE, fontweight='bold')
summary['mean_expression_correlation'] = round(corr_means, 4)

# Std scatter
axes[1].scatter(std_w, std_i, alpha=0.3, s=8, color=BLUE)
axes[1].set_xlabel('Western std', fontsize=11)
axes[1].set_ylabel('Indian std', fontsize=11)
axes[1].set_title('Gene Expression Variance\nWestern vs Indian', fontweight='bold')
corr_stds = float(np.corrcoef(std_w, std_i)[0, 1])
axes[1].text(0.05, 0.92, f'r = {corr_stds:.3f}',
             transform=axes[1].transAxes, fontsize=10, color=BLUE, fontweight='bold')
summary['std_expression_correlation'] = round(corr_stds, 4)

# Global distribution overlay
axes[2].hist(X_w_c.flatten()[::10], bins=80, alpha=0.55,
             color=BLUE, label='Western', density=True)
axes[2].hist(X_i_c.flatten()[::10], bins=80, alpha=0.55,
             color=RED,  label='Indian',  density=True)
axes[2].set_xlabel('Expression value', fontsize=11)
axes[2].set_ylabel('Density', fontsize=11)
axes[2].set_title('Global Expression Distribution', fontweight='bold')
axes[2].legend()

plt.suptitle('Expression Level Comparison — Western vs Indian Datasets',
             fontweight='bold', fontsize=13)
plt.tight_layout()
save('plot_02_expression_stats.png')

# ═══════════════════════════════════════════════════════════════
# 3. PCA — DO DATASETS SEPARATE IN GENE SPACE?
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 3 — PCA VISUALISATION")
print("=" * 60)

X_combined = np.vstack([X_w_c, X_i_c])
y_dataset  = np.array(['Western'] * len(X_w_c) + ['Indian'] * len(X_i_c))
y_label    = np.concatenate([y_w, y_i])

scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(X_combined)

pca        = PCA(n_components=50, random_state=42)
X_pca_full = pca.fit_transform(X_scaled)
X_pca      = X_pca_full[:, :2]

var_exp    = pca.explained_variance_ratio_
summary['pca_var_pc1'] = round(float(var_exp[0]*100), 2)
summary['pca_var_pc2'] = round(float(var_exp[1]*100), 2)

print(f"PC1 explains {var_exp[0]*100:.1f}%,  PC2 explains {var_exp[1]*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: coloured by dataset
for ds, color in [('Western', BLUE), ('Indian', RED)]:
    mask = y_dataset == ds
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, alpha=0.4, s=12, label=ds, rasterized=True)
axes[0].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)', fontsize=11)
axes[0].set_title('PCA — Coloured by Dataset', fontweight='bold')
axes[0].legend()

# Right: coloured by label
label_colors = {0: GREEN, 1: PURPLE}
label_names  = {0: 'Normal', 1: 'Tumor'}
for lab in [0, 1]:
    mask = y_label == lab
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=label_colors[lab], alpha=0.35, s=10,
                    label=label_names[lab], rasterized=True)
axes[1].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)', fontsize=11)
axes[1].set_title('PCA — Coloured by Tumor/Normal Label', fontweight='bold')
axes[1].legend()

plt.suptitle('PCA of Combined Western + Indian Datasets',
             fontweight='bold', fontsize=13)
plt.tight_layout()
save('plot_03_pca.png')

# ═══════════════════════════════════════════════════════════════
# 4. UMAP (if installed)
# ═══════════════════════════════════════════════════════════════
if HAS_UMAP:
    print("\n" + "=" * 60)
    print("STEP 4 — UMAP VISUALISATION")
    print("=" * 60)

    reducer  = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
    X_umap   = reducer.fit_transform(X_scaled)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ds, color in [('Western', BLUE), ('Indian', RED)]:
        mask = y_dataset == ds
        axes[0].scatter(X_umap[mask, 0], X_umap[mask, 1],
                        c=color, alpha=0.4, s=10, label=ds, rasterized=True)
    axes[0].set_title('UMAP — Coloured by Dataset', fontweight='bold')
    axes[0].legend()
    axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')

    for lab in [0, 1]:
        mask = y_label == lab
        axes[1].scatter(X_umap[mask, 0], X_umap[mask, 1],
                        c=label_colors[lab], alpha=0.35, s=10,
                        label=label_names[lab], rasterized=True)
    axes[1].set_title('UMAP — Coloured by Tumor/Normal', fontweight='bold')
    axes[1].legend()
    axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')

    plt.suptitle('UMAP of Combined Western + Indian Datasets',
                 fontweight='bold', fontsize=13)
    plt.tight_layout()
    save('plot_04_umap.png')
else:
    summary['umap_skipped'] = True

# ═══════════════════════════════════════════════════════════════
# 5. MANN-WHITNEY U TEST — which genes differ significantly?
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 5 — MANN-WHITNEY U TEST (per gene)")
print("=" * 60)

mw_pvals  = []
mw_stats  = []
ks_pvals  = []

for j in range(N_GENES):
    stat, p = mannwhitneyu(X_w_c[:, j], X_i_c[:, j], alternative='two-sided')
    mw_pvals.append(p)
    mw_stats.append(stat)
    _, p_ks = ks_2samp(X_w_c[:, j], X_i_c[:, j])
    ks_pvals.append(p_ks)

mw_pvals   = np.array(mw_pvals)
ks_pvals   = np.array(ks_pvals)

# Bonferroni correction
alpha_corrected = 0.05 / N_GENES
sig_mw  = (mw_pvals < alpha_corrected).sum()
sig_ks  = (ks_pvals < alpha_corrected).sum()

print(f"Genes significantly different (Mann-Whitney, Bonferroni): {sig_mw} / {N_GENES}")
print(f"Genes significantly different (KS test, Bonferroni)     : {sig_ks} / {N_GENES}")

summary['sig_genes_mannwhitney'] = int(sig_mw)
summary['sig_genes_ks']          = int(sig_ks)
summary['bonferroni_alpha']      = float(alpha_corrected)
summary['pct_sig_mannwhitney']   = round(sig_mw / N_GENES * 100, 1)

# ── Plot 5: Volcano-style p-value distribution ────────────────
log2fc = np.log2((mean_i + 1e-6) / (mean_w + 1e-6))
neg_log_p = -np.log10(mw_pvals + 1e-300)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Volcano plot
sig_mask = mw_pvals < alpha_corrected
axes[0].scatter(log2fc[~sig_mask], neg_log_p[~sig_mask],
                alpha=0.3, s=8, color=GRAY, label='Not significant')
axes[0].scatter(log2fc[sig_mask], neg_log_p[sig_mask],
                alpha=0.7, s=10, color=RED, label=f'Significant (n={sig_mw})')
axes[0].axhline(-np.log10(alpha_corrected), color=AMBER, lw=1.5,
                linestyle='--', label=f'Bonferroni threshold')
axes[0].axvline(0, color='gray', lw=0.8, linestyle=':')
axes[0].set_xlabel('Log₂ Fold Change (Indian / Western)', fontsize=11)
axes[0].set_ylabel('-log₁₀(p-value)', fontsize=11)
axes[0].set_title('Volcano Plot\nIndian vs Western Gene Expression',
                  fontweight='bold')
axes[0].legend(fontsize=9)

# Log2FC distribution
axes[1].hist(log2fc, bins=60, color=PURPLE, alpha=0.8, edgecolor='none')
axes[1].axvline(0, color='red', lw=2, linestyle='--', label='No change')
axes[1].set_xlabel('Log₂ Fold Change (Indian / Western)', fontsize=11)
axes[1].set_ylabel('Number of genes', fontsize=11)
axes[1].set_title('Distribution of Log₂ Fold Changes\nAcross All Common Genes',
                  fontweight='bold')
axes[1].legend()

med_fc = float(np.median(log2fc))
axes[1].text(0.62, 0.88, f'Median log₂FC = {med_fc:.3f}',
             transform=axes[1].transAxes, fontsize=10, color=PURPLE)

summary['median_log2fc']          = round(med_fc, 4)
summary['pct_upregulated_indian'] = round((log2fc > 0).sum() / N_GENES * 100, 1)
summary['pct_downregulated_indian'] = round((log2fc < 0).sum() / N_GENES * 100, 1)

plt.suptitle('Statistical Differences Between Western and Indian Datasets',
             fontweight='bold', fontsize=13)
plt.tight_layout()
save('plot_05_volcano_foldchange.png')

# ═══════════════════════════════════════════════════════════════
# 6. TOP-30 GENE SELECTION PER DATASET (RF + SHAP)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 6 — TOP-30 GENE SELECTION PER DATASET")
print("=" * 60)

def get_top30_shap(X, y, gene_names, label='dataset', n_top=30, random_state=42):
    """Train RF → SHAP → return top-n gene names + importance series."""
    from imblearn.over_sampling import SMOTE
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )
    sc     = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    # SMOTE only if imbalanced
    if (y_tr==0).sum() > 0 and (y_tr==1).sum() / (y_tr==0).sum() > 2:
        try:
            sm        = SMOTE(random_state=random_state, k_neighbors=min(5, (y_tr==0).sum()-1))
            X_tr_s, y_tr = sm.fit_resample(X_tr_s, y_tr)
        except Exception:
            pass

    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=random_state, n_jobs=-1)
    rf.fit(X_tr_s, y_tr)

    n_explain  = min(300, len(X_te_s))
    explainer  = shap.TreeExplainer(rf)
    shap_vals  = explainer.shap_values(X_te_s[:n_explain])

    if isinstance(shap_vals, list):
        sv = shap_vals[1]
    elif shap_vals.ndim == 3:
        sv = shap_vals[:, :, 1]
    else:
        sv = shap_vals

    importance = pd.Series(np.abs(sv).mean(axis=0), index=gene_names).sort_values(ascending=False)
    print(f"  {label} — top-5: {importance.head(5).index.tolist()}")
    return importance.head(n_top).index.tolist(), importance

print("  Selecting top-30 genes for Western dataset...")
top30_w, imp_w = get_top30_shap(X_w_c, y_w, common_genes, label='Western')

print("  Selecting top-30 genes for Indian dataset...")
# Indian: if only tumor, use RF importances without SHAP classification
if len(np.unique(y_i)) < 2:
    print("  [INFO] Indian dataset has one class only — using RF importance vs Western mean")
    rf_i = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    X_combined_ci = np.vstack([X_w_c, X_i_c])
    y_dataset_bin  = np.array([0]*len(X_w_c) + [1]*len(X_i_c))
    rf_i.fit(StandardScaler().fit_transform(X_combined_ci), y_dataset_bin)
    imp_i_vals = pd.Series(rf_i.feature_importances_, index=common_genes).sort_values(ascending=False)
    top30_i    = imp_i_vals.head(30).index.tolist()
    imp_i      = imp_i_vals
    print(f"  Indian — top-5: {top30_i[:5]}")
else:
    top30_i, imp_i = get_top30_shap(X_i_c, y_i, common_genes, label='Indian')

# ── Overlap ───────────────────────────────────────────────────
set_w       = set(top30_w)
set_i       = set(top30_i)
overlap     = sorted(set_w & set_i)
only_w      = sorted(set_w - set_i)
only_i      = sorted(set_i - set_w)

summary['top30_overlap']    = len(overlap)
summary['top30_only_w']     = len(only_w)
summary['top30_only_i']     = len(only_i)
summary['overlap_genes']    = overlap
summary['western_only_top'] = only_w
summary['indian_only_top']  = only_i

print(f"\nTop-30 overlap  : {len(overlap)} genes  → {overlap[:10]}...")
print(f"Western only    : {len(only_w)} genes  → {only_w[:5]}")
print(f"Indian only     : {len(only_i)} genes  → {only_i[:5]}")

# ── Plot 6a: Venn-style bar chart ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Venn numbers as bar
venn_data   = [len(overlap), len(only_w), len(only_i)]
venn_labels = [f'Shared\n({len(overlap)} genes)', f'Western\nonly ({len(only_w)})',
               f'Indian\nonly ({len(only_i)})']
venn_colors = [PURPLE, BLUE, RED]
bars = axes[0].bar(venn_labels, venn_data, color=venn_colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, venn_data):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 str(val), ha='center', fontweight='bold', fontsize=13)
axes[0].set_title('Top-30 Gene Overlap\nWestern vs Indian', fontweight='bold')
axes[0].set_ylabel('Number of genes')
axes[0].set_ylim(0, 35)
axes[0].grid(axis='y', alpha=0.3)

# Right: SHAP importance side-by-side for top-20 shared or all top-30 western
plot_genes = (overlap + only_w)[:20] if len(overlap) > 0 else top30_w[:20]
imp_w_plot = imp_w.reindex(plot_genes).fillna(0)
imp_i_plot = imp_i.reindex(plot_genes).fillna(0)

x3  = np.arange(len(plot_genes))
w3  = 0.38
axes[1].barh(x3 + w3/2, imp_w_plot.values, w3, color=BLUE, alpha=0.8, label='Western')
axes[1].barh(x3 - w3/2, imp_i_plot.values, w3, color=RED,  alpha=0.8, label='Indian')
axes[1].set_yticks(x3)
axes[1].set_yticklabels(plot_genes, fontsize=8)
axes[1].set_xlabel('Mean |SHAP| importance')
axes[1].set_title('SHAP Importance per Gene\nWestern vs Indian (top-20)', fontweight='bold')
axes[1].legend()
axes[1].invert_yaxis()

plt.suptitle('Top-30 Gene Selection Comparison', fontweight='bold', fontsize=13)
plt.tight_layout()
save('plot_06_gene_overlap_shap.png')

# ── Plot 6b: Top-30 heatmap for each dataset ──────────────────
all_top = list(dict.fromkeys(top30_w + top30_i))[:40]

mean_w_top = pd.Series(X_w_c[:, [common_genes.index(g) for g in all_top]].mean(axis=0), index=all_top)
mean_i_top = pd.Series(X_i_c[:, [common_genes.index(g) for g in all_top]].mean(axis=0), index=all_top)

heatmap_df = pd.DataFrame({'Western mean': mean_w_top, 'Indian mean': mean_i_top})
heatmap_df['log2FC (I/W)'] = np.log2((heatmap_df['Indian mean'] + 1e-6) /
                                       (heatmap_df['Western mean'] + 1e-6))

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(heatmap_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            linewidths=0.4, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Mean Expression Heatmap\nTop Genes (Western ∪ Indian top-30)',
             fontweight='bold', pad=14)
plt.tight_layout()
save('plot_06b_expression_heatmap.png')

# ═══════════════════════════════════════════════════════════════
# 7. KL DIVERGENCE — how different are gene distributions?
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 7 — KL DIVERGENCE (top 20 most shifted genes)")
print("=" * 60)

def kl_divergence_hist(a, b, bins=50):
    """Approximate KL divergence using histograms."""
    all_vals = np.concatenate([a, b])
    edges    = np.linspace(all_vals.min(), all_vals.max(), bins+1)
    p, _     = np.histogram(a, bins=edges, density=True)
    q, _     = np.histogram(b, bins=edges, density=True)
    p = p + 1e-10; q = q + 1e-10
    p /= p.sum(); q /= q.sum()
    return float(np.sum(p * np.log(p / q)))

kl_divs = []
for j in range(N_GENES):
    kl = kl_divergence_hist(X_w_c[:, j], X_i_c[:, j])
    kl_divs.append(kl)

kl_series = pd.Series(kl_divs, index=common_genes).sort_values(ascending=False)
top20_kl  = kl_series.head(20)

summary['mean_kl_divergence']   = round(float(kl_series.mean()), 4)
summary['median_kl_divergence'] = round(float(kl_series.median()), 4)
summary['top_kl_gene']          = str(kl_series.index[0])
summary['top_kl_value']         = round(float(kl_series.iloc[0]), 4)

print(f"Mean KL divergence across all genes: {kl_series.mean():.4f}")
print(f"Top-5 most shifted genes: {kl_series.head(5).index.tolist()}")

# ── Plot 7: KL divergence bar + example distributions ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_kl = [RED if g in set_w | set_i else GRAY for g in top20_kl.index]
axes[0].barh(range(20), top20_kl.values[::-1], color=colors_kl[::-1], alpha=0.85)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(top20_kl.index[::-1], fontsize=9)
axes[0].set_xlabel('KL Divergence (W || I)', fontsize=11)
axes[0].set_title('Top-20 Genes by Distribution Shift\n(KL Divergence)', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_els = [Patch(color=RED, label='In top-30 of W or I'),
              Patch(color=GRAY, label='Not in top-30')]
axes[0].legend(handles=legend_els, fontsize=9)

# Show distribution of the #1 most shifted gene
top_gene = kl_series.index[0]
j_top    = common_genes.index(top_gene)
axes[1].hist(X_w_c[:, j_top], bins=50, alpha=0.6, color=BLUE,
             label='Western', density=True)
axes[1].hist(X_i_c[:, j_top], bins=50, alpha=0.6, color=RED,
             label='Indian',  density=True)
axes[1].set_title(f'Expression Distribution\nMost shifted gene: {top_gene}',
                  fontweight='bold')
axes[1].set_xlabel('Expression value', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].legend()

plt.suptitle('Distribution Shift Analysis — Western vs Indian',
             fontweight='bold', fontsize=13)
plt.tight_layout()
save('plot_07_kl_divergence.png')

# ═══════════════════════════════════════════════════════════════
# 8. CORRELATION STRUCTURE
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 8 — CORRELATION STRUCTURE")
print("=" * 60)

# Use top-30 western genes for the correlation heatmap
top_idx_w = [common_genes.index(g) for g in top30_w]
corr_w = pd.DataFrame(X_w_c[:, top_idx_w], columns=top30_w).corr()
corr_i = pd.DataFrame(X_i_c[:, top_idx_w], columns=top30_w).corr()

corr_diff = corr_w - corr_i
summary['mean_corr_diff'] = round(float(np.abs(corr_diff.values).mean()), 4)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, corr_mat, title, cmap in zip(
    axes,
    [corr_w, corr_i, corr_diff],
    ['Western\nCorrelation (top-30 genes)',
     'Indian\nCorrelation (top-30 genes)',
     'Difference\n(Western - Indian)'],
    ['RdBu_r', 'RdBu_r', 'PuOr']
):
    mask = np.zeros_like(corr_mat, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    sns.heatmap(corr_mat, mask=mask, cmap=cmap, center=0,
                ax=ax, cbar=True, square=True,
                xticklabels=False, yticklabels=False,
                vmin=-1, vmax=1)
    ax.set_title(title, fontweight='bold', pad=10)

plt.suptitle('Gene-Gene Correlation Structure Comparison',
             fontweight='bold', fontsize=13)
plt.tight_layout()
save('plot_08_correlation_structure.png')

# ═══════════════════════════════════════════════════════════════
# 9. CUMULATIVE VARIANCE (PCA scree) — how complex each dataset is
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 9 — PCA SCREE / CUMULATIVE VARIANCE")
print("=" * 60)

pca_w = PCA(n_components=min(50, X_w_c.shape[1], X_w_c.shape[0]-1))
pca_i = PCA(n_components=min(50, X_i_c.shape[1], X_i_c.shape[0]-1))

pca_w.fit(StandardScaler().fit_transform(X_w_c))
pca_i.fit(StandardScaler().fit_transform(X_i_c))

cumvar_w = np.cumsum(pca_w.explained_variance_ratio_) * 100
cumvar_i = np.cumsum(pca_i.explained_variance_ratio_) * 100

def pcs_for_variance(cumvar, target=80):
    idx = np.searchsorted(cumvar, target)
    return int(idx + 1)

pcs80_w = pcs_for_variance(cumvar_w, 80)
pcs80_i = pcs_for_variance(cumvar_i, 80)
summary['pcs_80pct_variance_western'] = pcs80_w
summary['pcs_80pct_variance_indian']  = pcs80_i

print(f"Western: {pcs80_w} PCs needed for 80% variance")
print(f"Indian : {pcs80_i} PCs needed for 80% variance")

fig, ax = plt.subplots(figsize=(9, 5))
n_show_w = min(50, len(cumvar_w))
n_show_i = min(50, len(cumvar_i))
ax.plot(range(1, n_show_w+1), cumvar_w[:n_show_w],
        color=BLUE, lw=2.5, label='Western', marker='o', markersize=3)
ax.plot(range(1, n_show_i+1), cumvar_i[:n_show_i],
        color=RED,  lw=2.5, label='Indian',  marker='s', markersize=3)
ax.axhline(80, color=AMBER, lw=1.5, linestyle='--', label='80% threshold')
ax.axvline(pcs80_w, color=BLUE, lw=1, linestyle=':', alpha=0.8)
ax.axvline(pcs80_i, color=RED,  lw=1, linestyle=':', alpha=0.8)
ax.set_xlabel('Number of Principal Components', fontsize=11)
ax.set_ylabel('Cumulative Variance Explained (%)', fontsize=11)
ax.set_title('PCA Scree Curve\nComplexity of Gene Expression Patterns',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
ax.text(pcs80_w + 0.3, 74, f'{pcs80_w} PCs', color=BLUE, fontsize=9)
ax.text(pcs80_i + 0.3, 70, f'{pcs80_i} PCs', color=RED,  fontsize=9)
plt.tight_layout()
save('plot_09_pca_scree.png')

# ═══════════════════════════════════════════════════════════════
# 10. SAVE ALL SUMMARY STATS TO JSON FOR WORD DOC
# ═══════════════════════════════════════════════════════════════
summary_path = os.path.join(OUT_DIR, 'summary_stats.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n→ Summary stats saved: summary_stats.json")

# Save gene lists as CSVs
pd.DataFrame({'gene': top30_w, 'dataset': 'Western'}).to_csv(
    os.path.join(OUT_DIR, 'top30_western.csv'), index=False)
pd.DataFrame({'gene': top30_i, 'dataset': 'Indian'}).to_csv(
    os.path.join(OUT_DIR, 'top30_indian.csv'), index=False)
pd.DataFrame({'gene': overlap, 'category': 'Shared'}).to_csv(
    os.path.join(OUT_DIR, 'overlap_genes.csv'), index=False)

kl_series.reset_index().rename(
    columns={'index':'gene', 0:'kl_divergence'}
).head(50).to_csv(os.path.join(OUT_DIR, 'top50_kl_genes.csv'), index=False)

print("\n" + "=" * 60)
print("PYTHON ANALYSIS COMPLETE")
print(f"All outputs saved to: {OUT_DIR}")
print("=" * 60)
print("\nNext step: run   node generate_report.js   to build the Word doc")

STEP 1 — LOADING DATASETS
Western  — samples: 6322  genes: 566
  Tumor(1): 5436  Normal(0): 886

Indian   — samples: 232  genes in file: 566
Common genes (intersection): 566
Indian   — Tumor(1): 116  Normal(0): 116

STEP 2 — BASIC EXPRESSION STATISTICS
  → saved: plot_02_expression_stats.png

STEP 3 — PCA VISUALISATION
PC1 explains 69.2%,  PC2 explains 11.7%
  → saved: plot_03_pca.png

STEP 4 — UMAP VISUALISATION
  → saved: plot_04_umap.png

STEP 5 — MANN-WHITNEY U TEST (per gene)
Genes significantly different (Mann-Whitney, Bonferroni): 158 / 566
Genes significantly different (KS test, Bonferroni)     : 560 / 566
  → saved: plot_05_volcano_foldchange.png

STEP 6 — TOP-30 GENE SELECTION PER DATASET
  Selecting top-30 genes for Western dataset...
  Western — top-5: ['PGA5', 'GABRA4', 'SLC36A2', 'FTMT', 'CPA1']
  Selecting top-30 genes for Indian dataset...
  Indian — top-5: ['HSD17B13', 'F10', 'LPL', 'BTNL9', 'RHOJ']

Top-30 overlap  : 3 genes  → ['COL10A1', 'CSN1S1', 'PGA5']...
Western

In [2]:
import json
from docx import Document
from docx.shared import Inches

# Load summary stats
with open('dataset_comparison/summary_stats.json', 'r') as f:
    summary = json.load(f)

# Create a new Document
doc = Document()
doc.add_heading('Dataset Comparison Report: Western vs Indian Gene Expression', 0)

# Add sections
doc.add_heading('Dataset Overview', level=1)
doc.add_paragraph(f"Western Dataset: {summary['western_samples']} samples ({summary['western_tumor']} tumor, {summary['western_normal']} normal)")
doc.add_paragraph(f"Indian Dataset: {summary['indian_samples']} samples ({summary['indian_tumor']} tumor, {summary['indian_normal']} normal)")
doc.add_paragraph(f"Common Genes: {summary['common_genes']}")
doc.add_paragraph(f"Western-only Genes: {summary['western_only_genes']}, Indian-only Genes: {summary['indian_only_genes']}")

doc.add_heading('Expression Statistics', level=1)
doc.add_paragraph(f"Western Global Mean: {summary['western_global_mean']}, Std: {summary['western_global_std']}")
doc.add_paragraph(f"Indian Global Mean: {summary['indian_global_mean']}, Std: {summary['indian_global_std']}")
doc.add_paragraph(f"Mean Expression Correlation: {summary['mean_expression_correlation']}")
doc.add_paragraph(f"Std Expression Correlation: {summary['std_expression_correlation']}")
doc.add_picture('dataset_comparison/plot_02_expression_stats.png', width=Inches(6))

doc.add_heading('PCA Analysis', level=1)
doc.add_paragraph(f"PC1 Variance: {summary['pca_var_pc1']}%, PC2 Variance: {summary['pca_var_pc2']}%")
doc.add_picture('dataset_comparison/plot_03_pca.png', width=Inches(6))

doc.add_heading('Statistical Differences', level=1)
doc.add_paragraph(f"Significant Genes (Mann-Whitney): {summary['sig_genes_mannwhitney']} ({summary['pct_sig_mannwhitney']}%)")
doc.add_paragraph(f"Median Log2 Fold Change: {summary['median_log2fc']}")
doc.add_paragraph(f"Percent Upregulated in Indian: {summary['pct_upregulated_indian']}%, Downregulated: {summary['pct_downregulated_indian']}%")
doc.add_picture('dataset_comparison/plot_05_volcano_foldchange.png', width=Inches(6))

doc.add_heading('Top-30 Gene Selection', level=1)
doc.add_paragraph(f"Overlap: {summary['top30_overlap']} genes")
doc.add_paragraph(f"Western-only: {summary['top30_only_w']} genes")
doc.add_paragraph(f"Indian-only: {summary['top30_only_i']} genes")
doc.add_picture('dataset_comparison/plot_06_gene_overlap_shap.png', width=Inches(6))
doc.add_picture('dataset_comparison/plot_06b_expression_heatmap.png', width=Inches(6))

doc.add_heading('Distribution Shift', level=1)
doc.add_paragraph(f"Mean KL Divergence: {summary['mean_kl_divergence']}")
doc.add_paragraph(f"Top KL Gene: {summary['top_kl_gene']} (value: {summary['top_kl_value']})")
doc.add_picture('dataset_comparison/plot_07_kl_divergence.png', width=Inches(6))

doc.add_heading('Correlation Structure', level=1)
doc.add_paragraph(f"Mean Correlation Difference: {summary['mean_corr_diff']}")
doc.add_picture('dataset_comparison/plot_08_correlation_structure.png', width=Inches(6))

doc.add_heading('Complexity Analysis', level=1)
doc.add_paragraph(f"PCs for 80% Variance - Western: {summary['pcs_80pct_variance_western']}, Indian: {summary['pcs_80pct_variance_indian']}")
doc.add_picture('dataset_comparison/plot_09_pca_scree.png', width=Inches(6))

# Save the document
doc.save('dataset_comparison/comparison_report.docx')
print("Word document saved as comparison_report.docx")

Word document saved as comparison_report.docx
